In [1]:
import os
import sys
from pathlib import Path

sys.path.append("../")

In [2]:
from src.common import tools

config = tools.load_config()

In [3]:
from src.data import download

data_path = Path("../data")

if os.path.exists(data_path / config["lichess_puzzles_dataset_name"]):
    print(
        f"Dataset {config['lichess_puzzles_dataset_name']} already exists at {data_path}. Skipping download."
    )
else:
    download.kaggle_download_data(
        config["lichess_puzzles_dataset_handle"],
        data_path,
        config["lichess_puzzles_dataset_name"],
    )

Dataset lichess_puzzles already exists at ../data. Skipping download.


In [4]:
import pandas as pd

puzzles_df = pd.read_parquet("../data/lichess_puzzles/train-00000-of-00003.parquet")

In [5]:
puzzles_df.head()

,PuzzleId,GameId,FEN,Moves,Rating,RatingDeviation,Popularity,NbPlays,Themes,OpeningTags
0,00008,787zsVup/black#48,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K b - ...,f2g3 e6e7 b2b1 b3c1 b1c1 h6c1,2037,77,95,9125,"[crushing, hangingPiece, long, middlegame]",None
1,0000D,F8M8OS71#53,5rk1/1p3ppp/pq3b2/8/8/1P1Q1N2/P4PPP/3R2K1 w - ...,d3d6 f8d8 d6d8 f6d8,1455,74,96,35537,"[advantage, endgame, short]",None
2,0008Q,MQSyb3KW#127,8/4R3/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 w - - 1 64,e7f7 f5e5 e2f1 e5e6,1356,78,91,750,"[advantage, endgame, rookEndgame, short]",None
3,0009B,4MWQCxQ6/black#32,r2qr1k1/b1p2ppp/pp4n1/P1P1p3/4P1n1/B2P2Pb/3NBP...,b6c5 e2g4 h3g4 d1g4,1103,74,88,604,"[advantage, middlegame, short]","[Kings_Pawn_Game, Kings_Pawn_Game_Leonardis_Va..."
4,000Pw,au2lCK5o#73,6k1/5p1p/4p3/4q3/3nN3/2Q3P1/PP3P1P/6K1 w - - 2 37,e4d2 d4e2 g1f1 e2c3,1542,75,92,621,"[crushing, endgame, fork, short]",None


In [6]:
import chess


def expand_puzzle_positions(puzzles_df, fen_col="FEN", moves_col="Moves"):
    """
    Expand chess puzzles into individual position-move pairs.

    Parameters:
    -----------
    puzzles_df : pd.DataFrame
        DataFrame containing chess puzzles with FEN positions and move sequences
    fen_col : str
        Name of the column containing FEN positions (default: 'FEN')
    moves_col : str
        Name of the column containing move sequences (default: 'Moves')

    Returns:
    --------
    pd.DataFrame
        DataFrame with columns 'fen' and 'move', one row per move in each puzzle
    """
    expanded_data = []

    for _, row in puzzles_df.iterrows():
        board = chess.Board(row[fen_col])

        # Split moves if it's a string, otherwise assume it's already a list
        if isinstance(row[moves_col], str):
            moves = row[moves_col].split()
        else:
            moves = row[moves_col]

        for move_uci in moves:
            # Store current position and move
            expanded_data.append({"fen": board.fen(), "move": move_uci})

            # Make the move to get to next position
            try:
                move = chess.Move.from_uci(move_uci)
                board.push(move)
            except ValueError:
                # Skip invalid moves
                continue

    return pd.DataFrame(expanded_data)


# Example usage:
# expanded_df = expand_puzzle_positions(puzzles_df)

In [7]:
new_df = expand_puzzle_positions(puzzles_df[:100])

In [ ]:
import torch

new_df["evaluation"] = None

In [ ]:
from src.preprocess.datasets import ChessDataset

dataset = ChessDataset(new_df)

In [ ]:
t = torch.empty(0)

if t.numel() == 0:
    print("Tensor is empty")
else:
    print("Tensor is not empty")

Tensor is empty
